#**Project**

In [ ]:
!pip install imbalanced-learn statsmodels -q
!pip install streamlit imbalanced-learn statsmodels pyngrok -q
!pip install scikit-fuzzy shap pmdarima anthropic -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ks_2samp

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
import skfuzzy as fuzz
import skfuzzy.cluster as skfc
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from scipy.special import softmax
import shap
import warnings
warnings.filterwarnings('ignore')

drift_log = []

In [ ]:
df = pd.read_csv("StudentPerformanceFactors.csv")
df.head().T

#1. Data Preparation

##1.1. Data Understanding

In [ ]:
df.sample(5).T

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe(include="all").T

In [ ]:
df.nunique()

In [ ]:
low  = df["Exam_Score"].quantile(0.33)
high = df["Exam_Score"].quantile(0.66)

df["Score_Category"] = pd.cut(
    df["Exam_Score"],
    bins=[0, low, high, 101],
    labels=["Low", "Medium", "High"]
)

print(df["Score_Category"].value_counts())
sns.countplot(x="Score_Category", data=df)
plt.show()

#1.2. Data Clean

### 1.2.1 Remove Duplicates

In [ ]:
df_clean = df.copy()

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"── 1.2.1  Duplicates removed: {before - len(df_clean)}")

### 1.2.2 Fix Column Names

In [ ]:
df_clean.columns = (df_clean.columns
                    .str.strip()
                    .str.replace(" ", "_")
                    .str.replace(r"[^A-Za-z0-9_]", "")
                    .str.title())

df_clean.columns = [c.lower() for c in df_clean.columns]
print("── 1.2.2  Cleaned Column Names:", df_clean.columns.tolist())

### 1.2.3 Correct Data Types

In [ ]:
cat_cols = ["parental_involvement","access_to_resources","extracurricular_activities",
            "motivation_level","internet_access","family_income","teacher_quality",
            "school_type","peer_influence","learning_disabilities",
            "parental_education_level","distance_from_home","gender","score_category"]
num_cols = ["hours_studied","attendance","sleep_hours","previous_scores",
            "tutoring_sessions","physical_activity","exam_score"]

for c in cat_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].astype("category")
for c in num_cols:
    if c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")
print("── 1.2.3  Data types corrected")

## 1.2.4 Handle Missing Values

In [ ]:
print("\n── 1.2.4  Missing Values ──")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
for c in num_cols:
    if c in df_clean.columns:
        df_clean[c].fillna(df_clean[c].median(), inplace=True)
for c in cat_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].cat.add_categories("Unknown")
        df_clean[c].fillna("Unknown", inplace=True)
print("   Missing values handled ")

## 1.2.5 Remove Invalid Values

In [ ]:
df_clean = df_clean[(df_clean["exam_score"].between(0, 100)) &
                    (df_clean["attendance"].between(0, 100)) &
                    (df_clean["hours_studied"] >= 0)]
print(f"── 1.2.5  Rows after removing invalid values: {len(df_clean)}")

## 1.2.6 Standardize Categorical Values

In [ ]:
for c in cat_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].astype(str).str.strip().str.title().astype("category")
print("── 1.2.6  Categorical values standardized ")

## 1.2.7 Fix Inconsistent Units  (all numeric already consistent)

In [ ]:
print("── 1.2.7  Units are consistent ")

## 1.2.8 Outlier Flagging (IQR)

In [ ]:
print("\n── 1.2.8  Outlier Flagging (IQR) ──")
for c in num_cols:
    if c not in df_clean.columns: continue
    Q1, Q3 = df_clean[c].quantile(0.25), df_clean[c].quantile(0.75)
    IQR = Q3 - Q1
    flag = ((df_clean[c] < Q1 - 1.5*IQR) | (df_clean[c] > Q3 + 1.5*IQR))
    df_clean[f"{c}_outlier"] = flag
    print(f"   {c}: {flag.sum()} outliers flagged")

print(f"\n Clean dataset shape: {df_clean.shape}")

# ▸ 1.3  EDA

## 1.3.1  Univariate Analysis

### 1.3.1.1 Distribution of numerical features

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, c in enumerate(num_cols):
    if c not in df_clean.columns: continue
    axes[i].hist(df_clean[c].dropna(), bins=30,
                 color="#3498db", edgecolor="white", alpha=0.85)
    axes[i].set_title(f"Distribution: {c}")
    axes[i].set_xlabel(c); axes[i].set_ylabel("Frequency")
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle("1.3.1.1 – Numerical Features Distribution", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### 1.3.1.2 Count plots for categorical features


In [ ]:
cat_plot = [c for c in cat_cols if c not in ["score_category"] and c in df_clean.columns]
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, c in enumerate(cat_plot):
    order = df_clean[c].value_counts().index
    sns.countplot(data=df_clean, x=c, ax=axes[i], order=order, palette="Set2")
    axes[i].set_title(f"{c}"); axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=30)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle("1.3.1.2 – Categorical Features Count Plots", fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# 1.3.1.3 Skewness & Kurtosis
print("── 1.3.1.3  Skewness & Kurtosis ──")
sk_kurt = pd.DataFrame({
    "Skewness":  df_clean[num_cols].skew(),
    "Kurtosis":  df_clean[num_cols].kurt()
})
print(sk_kurt)

In [ ]:
# 1.3.1.4 Log transform for skewed features
skewed = sk_kurt[abs(sk_kurt["Skewness"]) > 0.75].index.tolist()
print(f"\n── 1.3.1.4  Applying log to skewed features: {skewed}")
for c in skewed:
    if c in df_clean.columns and (df_clean[c] > 0).all():
        df_clean[f"{c}_log"] = np.log1p(df_clean[c])
        plt.figure(figsize=(8, 3))
        plt.subplot(1,2,1); plt.hist(df_clean[c], bins=30, color="#e74c3c", alpha=0.8)
        plt.title(f"Original: {c}")
        plt.subplot(1,2,2); plt.hist(df_clean[f"{c}_log"], bins=30, color="#2ecc71", alpha=0.8)
        plt.title(f"Log: {c}")
        plt.tight_layout(); plt.show()

## 1.3.2  Bivariate Analysis

### 1.3.2.1 Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df_clean[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title("1.3.2.1 – Correlation Matrix Heatmap")
plt.tight_layout(); plt.show()

### 1.3.2.2 Scatter plots (top 4 correlations with exam_score)

In [ ]:
top_feats = corr["exam_score"].drop("exam_score").abs().nlargest(4).index
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, c in enumerate(top_feats):
    axes[i].scatter(df_clean[c], df_clean["exam_score"],
                    alpha=0.3, color="#9b59b6", s=10)
    m, b = np.polyfit(df_clean[c].dropna(), df_clean.loc[df_clean[c].notna(), "exam_score"], 1)
    x_line = np.linspace(df_clean[c].min(), df_clean[c].max(), 100)
    axes[i].plot(x_line, m*x_line+b, "r-", linewidth=2)
    axes[i].set_xlabel(c); axes[i].set_ylabel("Exam Score")
    axes[i].set_title(f"{c} vs Exam Score")
plt.suptitle("1.3.2.2 – Scatter Plots", fontsize=14)
plt.tight_layout(); plt.show()

### 1.3.2.3 Boxplot (categorical vs numerical)


In [ ]:
box_cats = ["parental_involvement","motivation_level","teacher_quality"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, c in enumerate(box_cats):
    if c in df_clean.columns:
        sns.boxplot(data=df_clean, x=c, y="exam_score", ax=axes[i], palette="Set3")
        axes[i].set_title(f"{c} vs Exam Score")
        axes[i].tick_params(axis="x", rotation=20)
plt.suptitle("1.3.2.3 – Boxplots (Categorical vs Numerical)", fontsize=14)
plt.tight_layout(); plt.show()

### 1.3.2.4 & 1.3.2.5 Barplot (categorical vs target)


In [ ]:
bar_cats = ["internet_access","school_type","peer_influence","gender"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
for i, c in enumerate(bar_cats):
    if c in df_clean.columns:
        grp = df_clean.groupby(c)["exam_score"].mean().sort_values(ascending=False)
        grp.plot(kind="bar", ax=axes[i], color=sns.color_palette("muted", len(grp)),
                 edgecolor="black")
        axes[i].set_title(f"Avg Exam Score by {c}")
        axes[i].set_ylabel("Mean Exam Score")
        axes[i].tick_params(axis="x", rotation=25)
plt.suptitle("1.3.2.4/5 – Barplots (Categorical vs Target)", fontsize=14)
plt.tight_layout(); plt.show()


## 1.3.3  Multivariate Analysis

### 1.3.3.1 3D Scatter Plot


In [ ]:
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection="3d")
sc  = ax.scatter(df_clean["hours_studied"],
                 df_clean["attendance"],
                 df_clean["exam_score"],
                 c=df_clean["exam_score"], cmap="plasma", alpha=0.4, s=8)
ax.set_xlabel("Hours Studied"); ax.set_ylabel("Attendance"); ax.set_zlabel("Exam Score")
ax.set_title("1.3.3.1 – 3D Scatter Plot")
plt.colorbar(sc, ax=ax, label="Exam Score")
plt.tight_layout(); plt.show()

### 1.3.3.2 VIF – Multicollinearity Check


In [ ]:
print("── 1.3.3.2  VIF – Multicollinearity Check ──")
vif_df_input = df_clean[num_cols].dropna()
vif_data = pd.DataFrame({
    "Feature": vif_df_input.columns,
    "VIF":     [variance_inflation_factor(vif_df_input.values, i)
                for i in range(vif_df_input.shape[1])]
})
print(vif_data.sort_values("VIF", ascending=False))

### 1.3.3.3 PCA


In [ ]:
print("\n── 1.3.3.3  PCA ──")
scaler_pca = StandardScaler()
X_pca_raw  = scaler_pca.fit_transform(df_clean[num_cols].dropna())
pca        = PCA()
pca.fit(X_pca_raw)
exp_var    = np.cumsum(pca.explained_variance_ratio_)
n_comp     = np.argmax(exp_var >= 0.95) + 1
print(f"   Components to explain 95% variance: {n_comp}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(exp_var)+1), exp_var, "bo-", linewidth=2)
plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
plt.xlabel("Number of Components"); plt.ylabel("Cumulative Explained Variance")
plt.title("1.3.3.3 – PCA Explained Variance"); plt.legend()
plt.tight_layout(); plt.show()

#  ▸ 1.4  DATA PREPROCESSING

In [ ]:
df_prep = df_clean.copy()

outlier_flag_cols = [c for c in df_prep.columns if c.endswith("_outlier")]
df_prep.drop(columns=outlier_flag_cols, inplace=True)


log_cols = [c for c in df_prep.columns if c.endswith("_log")]
df_prep.drop(columns=log_cols, inplace=True)

## 1.4.1 Encoding Categorical Variables

In [ ]:
print("── 1.4.1  Label Encoding ──")
le = LabelEncoder()
encode_cols = [c for c in cat_cols if c != "score_category" and c in df_prep.columns]
for c in encode_cols:
    df_prep[c] = le.fit_transform(df_prep[c].astype(str))

df_prep["score_category"] = le.fit_transform(df_prep["score_category"].astype(str))
print("   Encoding done ")

### 1.4.2 Feature Scaling

In [ ]:
print("── 1.4.2  Standard Scaling ──")
scaler  = StandardScaler()
scale_c = [c for c in num_cols if c != "exam_score" and c in df_prep.columns]
df_prep[scale_c] = scaler.fit_transform(df_prep[scale_c])
print("   Scaling done ")

### 1.4.3 Outlier Detection & Treatment (clip using IQR on exam_score)

In [ ]:
print("── 1.4.3  Outlier Treatment (Winsorizing exam_score) ──")
Q1 = df_prep["exam_score"].quantile(0.25)
Q3 = df_prep["exam_score"].quantile(0.75)
IQR = Q3 - Q1
df_prep["exam_score"] = df_prep["exam_score"].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print("   Done ")


#STEP 2-A : DBSCAN — Noise Detection & Structure Check

In [ ]:
cluster_features = ['hours_studied', 'attendance', 'previous_scores',
                    'sleep_hours', 'tutoring_sessions']

X_clust = df_prep[cluster_features].values

nbrs = NearestNeighbors(n_neighbors=5).fit(X_clust)
distances, _ = nbrs.kneighbors(X_clust)
k_distances = np.sort(distances[:, 4])[::-1]
suggested_eps = round(float(np.percentile(k_distances, 90)), 2)

plt.figure(figsize=(9, 3))
plt.plot(k_distances, color='#3498db', linewidth=1.5)
plt.axhline(y=suggested_eps, color='red', linestyle='--',
            label=f'suggested eps = {suggested_eps}')
plt.title('K-Distance Curve — DBSCAN')
plt.xlabel('Points (sorted)'); plt.ylabel('5th Nearest Distance')
plt.legend(); plt.tight_layout(); plt.show()

dbscan    = DBSCAN(eps=suggested_eps, min_samples=5)
db_labels = dbscan.fit_predict(X_clust)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = list(db_labels).count(-1)

print(f" DBSCAN → {n_clusters_db} clusters detected")
print(f"   Noise points : {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")

noise_mask = db_labels != -1
clean_indices = df_prep.index[noise_mask]
print(f" Clean data for FCM : {len(clean_indices)} rows")

#STEP 2-B : FCM — Fuzzy C-Means Clustering

In [ ]:
X_clust_clean = df_prep.loc[clean_indices, cluster_features].values




print(f" DBSCAN → {n_clusters_db} clusters | noise = {n_noise}")
print(f" Clean data : {X_clust_clean.shape[0]} rows")


n_clusters = 3

fcm_centers, fcm_membership, _, _, _, _, fcm_fpc = skfc.cmeans(
    data    = X_clust_clean.T,
    c       = n_clusters,
    m       = 1.3,
    error   = 0.005,
    maxiter = 1000,
    seed    = 42
)

mu_matrix = fcm_membership.T
hard_labels_fcm = np.argmax(mu_matrix, axis=1)


exam_scores_clean = df_prep.loc[clean_indices, 'exam_score'].values

cluster_means = {}
for c in range(n_clusters):
    cluster_means[c] = exam_scores_clean[hard_labels_fcm == c].mean()

sorted_clusters = sorted(cluster_means, key=cluster_means.get)

label_map = {
    sorted_clusters[0]: 2,
    sorted_clusters[1]: 1,
    sorted_clusters[2]: 0
}

hard_labels_fcm = np.array([label_map[l] for l in hard_labels_fcm])


# ── إضافة النتائج للـ DataFrame ──
df_prep.loc[clean_indices, 'FCM_Cluster'] = hard_labels_fcm
df_prep.loc[clean_indices, 'mu_low'] = mu_matrix[:, sorted_clusters[0]]
df_prep.loc[clean_indices, 'mu_medium'] = mu_matrix[:, sorted_clusters[1]]
df_prep.loc[clean_indices, 'mu_high'] = mu_matrix[:, sorted_clusters[2]]


print(f"\n FCM converged | FPC = {fcm_fpc:.4f}")
print("\n── توزيع الطلاب ──")

for i, name in enumerate(['Low Risk', 'Medium Risk', 'High Risk']):
    count = (hard_labels_fcm == i).sum()
    avg_e = exam_scores_clean[hard_labels_fcm == i].mean()
    print(f"{name:12s}: {count:4d} ({count/len(hard_labels_fcm)*100:.1f}%)"
          f" | exam = {avg_e:.2f}")


X_full_clean = df_prep.loc[clean_indices, cluster_features].values

fcm_centers_full = np.zeros((n_clusters, X_full_clean.shape[1]))

for c in range(n_clusters):
    weights = mu_matrix[:, sorted_clusters[c]]
    fcm_centers_full[c] = np.average(X_full_clean, axis=0, weights=weights)

np.save('fcm_centers.npy', fcm_centers_full)

print(f"\nCenters saved | shape = {fcm_centers_full.shape}")

#STEP 2-C : Visualize FCM Clusters

In [ ]:
pca2       = PCA(n_components=2, random_state=42)
X_2d_clean = pca2.fit_transform(X_clust_clean)

colors      = ['#2ecc71', '#f39c12', '#e74c3c']
labels_viz  = ['Low Risk', 'Medium Risk', 'High Risk']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(n_clusters):
    mask = hard_labels_fcm == i
    axes[0].scatter(X_2d_clean[mask, 0], X_2d_clean[mask, 1],
                    c=colors[i], label=labels_viz[i], alpha=0.5, s=12)
axes[0].set_title(f'FCM Hard Labels (PCA 2D) | FPC={fcm_fpc:.3f}')
axes[0].legend(); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

uncertainty = 1 - mu_matrix.max(axis=1)
sc = axes[1].scatter(X_2d_clean[:, 0], X_2d_clean[:, 1],
                     c=uncertainty, cmap='RdYlGn_r', alpha=0.6, s=12)
plt.colorbar(sc, ax=axes[1], label='Uncertainty')
axes[1].set_title('FCM Membership Uncertainty')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.suptitle('FCM Clustering Results', fontsize=13)
plt.tight_layout(); plt.show()

print(f" FPC = {fcm_fpc:.4f} — Excellent✓")


#  STEP 3: SVM CLASSIFICATION


In [ ]:

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib


print("📊 Preparing data for SVM training...")


feature_cols = [c for c in df_prep.columns
                if c not in ['exam_score', 'score_category', 'FCM_Cluster',
                             'mu_low', 'mu_medium', 'mu_high']]

X = df_prep[feature_cols].copy()
y = df_prep['FCM_Cluster'].copy()


mask = y.notna()
X = X[mask]
y = y[mask]

print(f"   ✅ Total students: {len(X)}")
print(f"   ✅ Number of features: {len(feature_cols)}")

# 2. Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Data Split:")
print(f"   Training: {len(X_train)} students")
print(f"   Testing: {len(X_test)} students")


print("\n Training SVM...")

svm_model = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm_model.fit(X_train, y_train)

print("   ✅ Training completed successfully!")

# 4. Evaluate the model
y_pred = svm_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n📈 Model Performance:")
print(f"   Accuracy: {accuracy:.2%}")

# 5. Detailed classification report
print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred,
                           target_names=['🟢 Low Risk', '🟡 Medium Risk', '🔴 High Risk']))

# 6. Confusion Matrix (shows where model is right/wrong)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.xlabel('📌 Predicted', fontsize=12)
plt.ylabel('✅ Actual', fontsize=12)
plt.title('SVM Confusion Matrix - Model Performance', fontsize=14)
plt.tight_layout()
plt.show()

# 7. Save model for future use
joblib.dump(svm_model, 'svm_model.pkl')
print(f"\n💾 Model saved as: svm_model.pkl")

## SVM Confidence Gap (Boundary Distance)
### Objective: Measuring the classifier's certainty by calculating the distance from the decision boundary.

In [ ]:
print("📊 Calculating SVM Confidence Gaps...")

probs = svm_model.predict_proba(X_test)

# الفرق بين أعلى احتمال وثاني أعلى احتمال
sorted_probs = -np.sort(-probs, axis=1)

svm_gaps = sorted_probs[:, 0] - sorted_probs[:, 1]

X_test_indices = X_test.index

df_prep.loc[X_test_indices, 'SVM_Gap'] = svm_gaps

print(f"✅ SVM Gap calculated for {len(X_test)} students.")
print(f"\n📌 Sample Gaps:")
print(df_prep.loc[X_test_indices, 'SVM_Gap'].head())



#STEP 4-A: SHAP

In [ ]:

import shap
import warnings
warnings.filterwarnings('ignore')

print(" Calculating SHAP values to explain model decisions...")


sample_size = min(100, len(X_test))
X_sample = X_test.iloc[:sample_size].copy()

print(f"   ✅ Explaining {sample_size} students")


print("    Creating explainer...")
explainer = shap.KernelExplainer(svm_model.predict_proba, X_train.iloc[:200])

print("    Computing SHAP values...")
shap_values = explainer.shap_values(X_sample)

print("   ✅ SHAP values calculated successfully!")


print("\n Generating SHAP Summary Plot...")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
plt.title('SHAP Feature Importance - What affects student risk most?', fontsize=14)
plt.tight_layout()
plt.show()


print("\n📊 Generating SHAP Bar Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, plot_type="bar", show=False)
plt.title('Average Feature Importance (SHAP)', fontsize=14)
plt.tight_layout()
plt.show()


print("\n🔍 Explaining a sample student's prediction:")
student_idx = 0

# ==================== التنبؤ + الاحتمالات ====================
pred = int(svm_model.predict(X_sample.iloc[[student_idx]])[0])
proba = svm_model.predict_proba(X_sample.iloc[[student_idx]])[0]

risk_names = ['🟢 Low Risk', '🟡 Medium Risk', '🔴 High Risk']

print(f"\n   📌 Student {student_idx}:")
print(f"      Model Prediction: {risk_names[pred]} (Class {pred})")
print(f"      Probabilities → Low: {proba[0]:.4f} | Medium: {proba[1]:.4f} | High: {proba[2]:.4f}\n")

# ==================== SHAP Values للكلاس المتوقع ====================
shap_student = shap_values[pred][student_idx]

print(f"   🔑 Top 10 reasons for this decision (Class {pred} - {risk_names[pred]}):")
print(f"   {'='*70}")

# أهم 10 متغيرات
top_indices = np.argsort(np.abs(shap_student))[-10:][::-1]

for idx in top_indices:
    feature_name = feature_cols[idx]
    feature_value = X_sample.iloc[student_idx][feature_name]
    shap_impact = shap_student[idx]

    if shap_impact > 0:
        arrow = "⬆️ INCREASES risk (pushes toward High Risk)"
        color = "🔴"
    else:
        arrow = "⬇️ DECREASES risk (pushes toward Low Risk)"
        color = "🟢"

    print(f"      {color} {feature_name:28} = {feature_value:.3f}    |    Impact: {shap_impact:+.5f}")
    print(f"         → {arrow}")
    print("-" * 65)

# ==================== Waterfall Plot ====================
print(f"\n📊 Generating Waterfall Plot for Student {student_idx}...")

exp = shap.Explanation(
    values=shap_student,
    base_values=explainer.expected_value[pred],
    data=X_sample.iloc[student_idx].values,
    feature_names=feature_cols
)

shap.waterfall_plot(exp, show=False)
plt.title(f'Waterfall Plot - Why Student {student_idx} was classified as {risk_names[pred]}', fontsize=13)
plt.tight_layout()
plt.show()

# ==================== حفظ الـ SHAP Values ====================
import joblib
joblib.dump(shap_values, 'shap_values.pkl')
print(f"\n💾 SHAP values saved as: shap_values.pkl")

print("\n✅ STEP 4A (SHAP) completed successfully!")


## STEP 5 — SHAP Cluster Reliability Setup


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("📊 Initializing Integrated SHAP Reliability Engine...")

cluster_shap_means = {}
n_clusters = 3

for i in range(n_clusters):

    mask = (y_test.values[:sample_size] == i)
    if mask.any():

        cluster_shap_means[i] = np.mean(shap_values[i], axis=0)

def get_shap_reliability(student_idx, predicted_cluster):
    """
    تستخدم بصمة الـ SHAP للطالب الحالي لتقييم مدى مطابقتها لمتوسط الفئة.
    تتعامل بذكاء مع الـ Indexing لضمان عدم حدوث Error أثناء الـ Full Evaluation.
    """
    try:

        student_vector = shap_values[predicted_cluster][student_idx].reshape(1, -1)
        mean_vector = cluster_shap_means[predicted_cluster].reshape(1, -1)

        return float(cosine_similarity(student_vector, mean_vector)[0][0])
    except Exception as e:
        return 0.5

print(f"✅ SHAP Reliability setup complete. Sync verified for ReAct Loop.")

SARIMA Forecasting

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("\n📈 STEP 6 — SARIMA Forecasting")

cluster_series = {
    0: df_prep.loc[clean_indices, 'mu_low'].reset_index(drop=True),
    1: df_prep.loc[clean_indices, 'mu_medium'].reset_index(drop=True),
    2: df_prep.loc[clean_indices, 'mu_high'].reset_index(drop=True)
}

cluster_names = {
    0: "🟢 Low Risk",
    1: "🟡 Medium Risk",
    2: "🔴 High Risk"
}

trend_multipliers = {}

for cluster_id, series in cluster_series.items():

    print("\n" + "="*50)
    print(f"📊 {cluster_names[cluster_id]}")
    print("="*50)

    model = SARIMAX(
        series,
        order=(1,1,1),
        seasonal_order=(1,1,1,12),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit(disp=False)

    forecast = results.forecast(steps=10)

    trend_slope = forecast.iloc[-1] - forecast.iloc[0]

    if trend_slope > 0:

        trend_multiplier = 1.2
        trend_name = "Increasing Risk"

    elif trend_slope < 0:

        trend_multiplier = 0.8
        trend_name = "Decreasing Risk"

    else:

        trend_multiplier = 1.0
        trend_name = "Stable"

    trend_multipliers[cluster_id] = trend_multiplier

    print(f"\n📈 Trend: {trend_name}")
    print(f"🔥 Multiplier: {trend_multiplier}")

    # رسم التريند
    plt.figure(figsize=(10,4))

    plt.plot(
        series[-50:],
        label='Historical',
        linewidth=2
    )

    plt.plot(
        range(len(series[-50:]), len(series[-50:])+10),
        forecast,
        label='Forecast',
        linewidth=2
    )

    plt.title(f"SARIMA Forecast — {cluster_names[cluster_id]}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

print("\n✅ SARIMA completed successfully!")

μ(predicted)

In [ ]:
print("\n📊 STEP 7 — μ(predicted)")

if pred == 0:

    mu_pred = df_prep.loc[
        X_sample.index[student_idx],
        'mu_low'
    ]

elif pred == 1:

    mu_pred = df_prep.loc[
        X_sample.index[student_idx],
        'mu_medium'
    ]

else:

    mu_pred = df_prep.loc[
        X_sample.index[student_idx],
        'mu_high'
    ]

print(f"\n✅ μ(predicted) = {mu_pred:.4f}")

Severity Score

In [ ]:
print("\n🚨 STEP 8 — Severity Score")

severity_map = {
    0: 0.2,
    1: 0.6,
    2: 1.0
}

severity_score = severity_map[pred]

print(f"\n✅ Severity Score = {severity_score:.4f}")

SVM Gap

In [ ]:
print("\n📊 STEP 9 — SVM Gap")

SVM_Gap = df_prep.loc[
    X_sample.index[student_idx],
    'SVM_Gap'
]

print(f"\n✅ SVM Gap = {SVM_Gap:.4f}")

Intelligent Score

In [ ]:
print("\n🧠 STEP 10 — Intelligent Score")

trend_multiplier = trend_multipliers[pred]


shap_similarity = get_shap_reliability(student_idx, pred)

intelligent_score = (
    0.25 * severity_score +
    0.20 * SVM_Gap +
    0.25 * mu_pred +
    0.15 * trend_multiplier +
    0.15 * shap_similarity
)

print("\n🔥 FINAL INTELLIGENT SCORE")
print("="*50)

print(f"Severity Score    : {severity_score:.4f}")
print(f"SVM Gap           : {SVM_Gap:.4f}")
print(f"μ(predicted)      : {mu_pred:.4f}")
print(f"Trend Mult.       : {trend_multiplier:.4f}")
print(f"SHAP Reliability  : {shap_similarity:.4f}")

print("-"*50)
print(f"🧠 Intelligent Score = {intelligent_score:.4f}")

Final Decision

In [ ]:
print("\n🚨 STEP 11 — Final Decision")

if intelligent_score >= 0.75:

    final_decision = "🔴 HIGH RISK"

elif intelligent_score >= 0.45:

    final_decision = "🟡 MEDIUM RISK"

else:

    final_decision = "🟢 LOW RISK"

print(f"\n✅ Final Decision = {final_decision}")

#RAG + ReAct Loop Implementation
## Part 1: Explicit RAG | Part 2: Implicit RAG | Part 3: ReAct 7-Path Decision Loop
                                  

## Constants

In [ ]:
RISK_NAMES = {0: "LOW RISK", 1: "MEDIUM RISK", 2: "HIGH RISK"}
print("✅ RISK_NAMES defined")

##MASTER_INTERVENTION_DB

In [ ]:
# 1. القاموس الموحد - MASTER INTERVENTION DATABASE
MASTER_INTERVENTION_DB = {
    "attendance": {
        "target": "School Advisor",
        "title": "📅 Attendance Recovery Plan",
        "actions": ["Send automated alert", "Schedule counselor check-in", "Provide compensatory schedule"]
    },
    "hours_studied": {
        "target": "Academic Tutor",
        "title": "⏱️ Time Management Intervention",
        "actions": ["Deliver weekly study schedule", "Enroll in skills workshop", "Set portal reminders"]
    },
    "previous_scores": {
        "target": "Academic Coordinator",
        "title": "📊 Academic History Review",
        "actions": ["Conduct diagnostic assessment", "Design remediation plan", "Increase checkpoint frequency"]
    },
    "parental_involvement": {
        "target": "Family Support Unit",
        "title": "👪 Family Engagement Plan",
        "actions": ["Email family summary", "Schedule parent-teacher meeting", "Provide at-home guide"]
    },
    "motivation_level": {
        "target": "Student Mentor",
        "title": "🎯 Motivational Re-engagement",
        "actions": ["Goal-setting session", "Connect with peer mentor", "Gamified progress tracking"]
    },
    "sleep_hours": {
        "target": "Wellness Counselor",
        "title": "😴 Sleep & Wellness Support",
        "actions": ["Flag for health follow-up", "Share sleep-hygiene resources", "Adjust home schedule"]
    },
    "default": {
        "target": "Student Success Coach",
        "title": "🔍 General Academic Monitoring",
        "actions": ["Increase check-in frequency", "Review profile with advisor", "Standard risk package"]
    }
}

### Helpers

In [ ]:
# 2. فلاتر تحليل البيانات
def extract_dominant_factor(shap_dict):
    top_feat = max(shap_dict, key=shap_dict.get)
    return {'feature': top_feat, 'explanation': f"Strongest impact from '{top_feat}'."}

def calculate_agentic_confidence(rag_score, arima_trend):
    conf = rag_score
    urgency_mult = 1.0
    reason = "Stable trend."
    if arima_trend > 0.05:
        conf = min(conf * 1.2, 1.0); urgency_mult = 1.3
        reason = "Deteriorating trend (Escalated)."
    elif arima_trend < -0.05:
        conf = conf * 0.8; urgency_mult = 0.7
        reason = "Improvement trend (Dampened)."
    return conf, urgency_mult, reason, (conf < 0.4 or conf > 0.9)

###Agent Core

In [ ]:
# 3. المحرك الرئيسي (النسخة المختصرة)
def generate_agentic_action(student_idx, rag_score, arima_trend, shap_values, feature_cols, predicted_cluster):
    try:
        s_idx = int(np.array(student_idx).flatten()[0]) % 100
        c_idx = int(predicted_cluster)
        student_shap = shap_values[s_idx, :, c_idx] if not isinstance(shap_values, list) else shap_values[c_idx][s_idx]
    except:
        student_shap = shap_values[0]

    shap_dict = {f: float(v) for f, v in zip(feature_cols, np.array(student_shap).flatten())}

    dom = extract_dominant_factor(shap_dict)
    conf, mult, reason_log, human_req = calculate_agentic_confidence(rag_score, arima_trend)

    # سحب البيانات من المرجع (MASTER_INTERVENTION_DB)
    info = MASTER_INTERVENTION_DB.get(dom['feature'], MASTER_INTERVENTION_DB['default'])
    intensity = "HIGH" if (rag_score * mult) > 0.75 else "MEDIUM" if (rag_score * mult) > 0.4 else "LOW"

    return {
        "risk_level": "HIGH RISK" if rag_score > 0.7 else "MEDIUM RISK" if rag_score > 0.4 else "LOW RISK",
        "action_type": info['title'],
        "target_person": info['target'],
        "priority": "IMMEDIATE" if intensity == "HIGH" else "STANDARD",
        "intensity": intensity,
        "steps": info['actions'],
        "root_causes": [dom['feature']],
        "reasoning_path": {
            "shap_reason": dom['explanation'],
            "arima_logic": reason_log,
            "final_confidence": f"{conf:.2%}"
        },
        "requires_human_review": human_req
    }

###Router

In [ ]:
def route_and_act(student_idx, svm_pred, svm_proba, intelligent_score, arima_trend,
                  shap_values, feature_cols, config=None, verbose=True):
    if config is None: config = ROUTING_CONFIG

    svm_gap = float(np.sort(svm_proba)[-1] - np.sort(svm_proba)[-2])
    entropy_val = _entropy(svm_proba)

    # هنا بننادي على "عقل الأجينت" اللي في الخلية 3
    action_result = generate_agentic_action(student_idx, intelligent_score, arima_trend,
                                           shap_values, feature_cols, svm_pred)

    if entropy_val >= config["HIGH_UNCERTAINTY_ENTROPY"]:
        path, name = 7, "High Uncertainty"
    elif svm_gap < config["SVM_FAIL_GAP_THRESHOLD"]:
        path, name = 5, "Model Uncertainty Fallback"
    else:
        path, name = 1, "Confident Prediction"

    return _package(path, name, svm_pred, action_result['risk_level'],
                    intelligent_score, action_result, action_result['reasoning_path']['shap_reason'])

## Part 1 — Knowledge-Base Retrieval: Explicit Evidence Index

In [ ]:
def build_explicit_rag(X_sample, shap_values, n_neighbors=5):
    n_students = X_sample.shape[0]
    rag_vector_list = []

    for i in range(n_students):
        shap_all_classes = shap_values[i, :, :].flatten()
        feature_vec      = X_sample.iloc[i].values
        rag_vec          = np.concatenate([feature_vec, shap_all_classes])
        rag_vector_list.append(rag_vec)

    rag_vectors = np.array(rag_vector_list)

    nn_model = NearestNeighbors(
        n_neighbors = n_neighbors + 1,
        metric      = 'cosine',
        algorithm   = 'brute'
    )
    nn_model.fit(rag_vectors)

    print(f"✅ RAG Index built — {n_students} students, vector dim = {rag_vectors.shape[1]}")
    return nn_model, rag_vectors

print("✅ build_explicit_rag() defined")

In [ ]:
def retrieve_neighbors(student_idx, nn_model, rag_vectors, y_labels, k=5):
    query = rag_vectors[student_idx].reshape(1, -1)
    distances, indices = nn_model.kneighbors(query)

    raw_indices      = indices[0]
    raw_distances    = distances[0]
    raw_similarities = 1.0 - raw_distances

    mask = raw_indices != student_idx
    retrieved_indices      = raw_indices[mask][:k]
    retrieved_similarities = raw_similarities[mask][:k]
    retrieved_labels       = np.array(y_labels)[retrieved_indices]

    vote_counts          = Counter(retrieved_labels)
    retrieval_majority   = vote_counts.most_common(1)[0][0]
    retrieval_confidence = vote_counts.most_common(1)[0][1] / len(retrieved_labels)

    return (retrieved_indices, retrieved_labels,
            retrieved_similarities, retrieval_majority, retrieval_confidence)

print("✅ retrieve_neighbors() defined")

## Part 2 — Latent-Pattern Discovery: Implicit Similarity Mapping

In [ ]:
def compute_implicit_similarity(student_idx, predicted_cluster, shap_values, cluster_shap_means):
    student_shap_vector = shap_values[student_idx, :, predicted_cluster].reshape(1, -1)
    mean_shap_vector    = cluster_shap_means[predicted_cluster].reshape(1, -1)
    implicit_similarity = cosine_similarity(student_shap_vector, mean_shap_vector)[0][0]
    return float(implicit_similarity)

print("✅ compute_implicit_similarity() defined")

In [ ]:


def adjust_rag_with_trend(retrieval_confidence, trend_multiplier, cluster_id):
    """
    Adjust RAG confidence based on ARIMA trend direction (using trend_multiplier).
    trend_multiplier > 1.0 = rising risk (ARIMA votes WITH RAG for High Risk)
    trend_multiplier < 1.0 = declining risk (ARIMA votes AGAINST RAG)
    """
    if cluster_id == 2:  # High Risk cluster
        if trend_multiplier > 1.0:  # ARIMA votes WITH RAG (trend rising = risk increasing)
            adjusted = retrieval_confidence * 1.2
            return min(1.0, adjusted), "WITH_RAG", "increased"
        else:  # ARIMA votes AGAINST RAG (trend declining despite being high risk)
            adjusted = retrieval_confidence * 0.7
            return adjusted, "AGAINST_RAG", "decreased"

    elif cluster_id == 0:  # Low Risk cluster
        if trend_multiplier < 1.0:  # ARIMA votes WITH RAG (trend declining = staying low)
            adjusted = retrieval_confidence * 1.2
            return min(1.0, adjusted), "WITH_RAG", "increased"
        else:  # ARIMA votes AGAINST RAG (trend rising toward higher risk)
            adjusted = retrieval_confidence * 0.7
            return adjusted, "AGAINST_RAG", "decreased"

    # Medium Risk cluster - no adjustment
    return retrieval_confidence, "NEUTRAL", "unchanged"

##Sentinel-Protocol: Dual-Signal Stability Guardian

In [ ]:
# ══════════════════════════════════════════════════════════════
# fix 1-SENTINEL-PROTOCOL: DUAL-SIGNAL DRIFT DETECTION
# ═════════════════════════════════════════════════════════════-

def detect_dual_signal_drift(
    X_current,
    X_reference,
    svm_model,
    fcm_centers_full,
    confidence_drop_threshold  = 0.15,   # Signal A: SVM gap threshold
    centroid_shift_threshold   = 0.10,   # Signal B: centroid movement ratio
    ks_pvalue_threshold        = 0.05,   # Signal B: KS-test significance level
    verbose                    = True,
):
    # ── SIGNAL A: Model Confidence (SVM probability gap) ────────
    probs_current   = svm_model.predict_proba(X_current)
    sorted_probs    = np.sort(probs_current, axis=1)[:, ::-1]
    gaps_current    = sorted_probs[:, 0] - sorted_probs[:, 1]
    avg_gap_current = float(np.mean(gaps_current))

    probs_ref       = svm_model.predict_proba(X_reference)
    sorted_ref      = np.sort(probs_ref, axis=1)[:, ::-1]
    gaps_ref        = sorted_ref[:, 0] - sorted_ref[:, 1]
    avg_gap_ref     = float(np.mean(gaps_ref))

    # التعديل هنا: بنحسب الـ drop الفعلي
    confidence_drop = avg_gap_ref - avg_gap_current
    signal_a        = confidence_drop > confidence_drop_threshold

    # ── SIGNAL B: Data Distribution Drift ───────────────────────
    feature_names    = X_current.columns.tolist()
    drifted_features = []
    for col in feature_names:
        stat, pval = ks_2samp(X_reference[col].values, X_current[col].values)
        if pval < ks_pvalue_threshold:
            drifted_features.append(col)

    # Centroid movement logic (Safe truncated version)
    n_fcm_features = fcm_centers_full.shape[1]
    ref_values = X_reference.values[:, :n_fcm_features]
    current_values = X_current.values[:, :n_fcm_features]

    centroid_dists_ref = [float(np.mean(np.linalg.norm(ref_values - center, axis=1))) for center in fcm_centers_full]
    centroid_dists_curr = [float(np.mean(np.linalg.norm(current_values - center, axis=1))) for center in fcm_centers_full]

    shift_ratio = abs(np.mean(centroid_dists_curr) - np.mean(centroid_dists_ref)) / (np.mean(centroid_dists_ref) + 1e-9)
    signal_b     = (len(drifted_features) > 0) or (shift_ratio > centroid_shift_threshold)

    # ── DUAL GATE (The core logic) ───────────────────────────────
    # Retraining only if BOTH signals are active
    should_retrain = signal_a and signal_b

    if verbose:
        print("\n" + "═"*60)
        print("🛡️ SENTINEL REPORT: DUAL-SIGNAL DRIFT ANALYSIS")
        print("═"*60)
        status_a = "🚨 ALERT" if signal_a else "✅ STABLE"
        print(f"📡 SIGNAL A (Model Stability): {status_a}")
        print(f"   Confidence Drop: {confidence_drop:.4f} (Limit: {confidence_drop_threshold})")

        status_b = "🚨 ALERT" if signal_b else "✅ STABLE"
        print(f"📊 SIGNAL B (Data Integrity):  {status_b}")
        print(f"   Drifted Features: {len(drifted_features)} | Shift Ratio: {shift_ratio:.4f}")

        print("-" * 60)
        if should_retrain:
            print("🔴 DECISION: CRITICAL DRIFT DETECTED → EXECUTE RETRAINING")
        else:
            reason = "Dual-gate not met" if (signal_a or signal_b) else "System fully stable"
            print(f"🟢 DECISION: MAINTAIN CURRENT MODEL ({reason})")
        print("═"*60)

    return {
        "signal_a": signal_a,
        "signal_b": signal_b,
        "should_retrain": should_retrain,
        "avg_svm_gap": avg_gap_current,
        "centroid_shift": shift_ratio,
        "drifted_features": drifted_features,
    }

In [ ]:
# ── DEMO: Sentinel Drift Test ───────────────────
# تقسيم البيانات لمحاكاة "الوضع الحالي" مقابل "المرجع التاريخي"
split_point  = int(len(X_test) * 0.5)
X_reference  = X_test.iloc[:split_point]
X_current    = X_test.iloc[split_point:]

# تشغيل الفحص باستخدام الحارس (Sentinel)
drift_report = detect_dual_signal_drift(
    X_current         = X_current,
    X_reference       = X_reference,
    svm_model         = svm_model,
    fcm_centers_full  = fcm_centers_full,
    confidence_drop_threshold = 0.15,
    centroid_shift_threshold  = 0.10,
    ks_pvalue_threshold       = 0.05,
)

# اتخاذ القرار بناءً على الـ Dual Gate
if drift_report["should_retrain"]:
    print(f"\n🔄 [AGENT ACTION]: Critical drift in {len(drift_report['drifted_features'])} features detected.")
    print("👉 Re-triggering Pipeline: (DBSCAN ➔ FCM ➔ SVM Update)")
    # هنا تحطي الـ Function اللي بتنادي على الـ Retrain لو موجودة
else:
    print(f"\n✅ [AGENT STATUS]: Model remains optimal. Confidence Gap ({drift_report['avg_svm_gap']:.2f}) is within safety bounds.")
    print("👉 Maintaining current parameters.")

## Part 3 — ReAct Decision Loop (7 Paths)

In [ ]:
# ══════════════════════════════════════════════════════════════
# ⚡ UPGRADED 7-PATH ROUTING SYSTEM (Clean Agentic Version)
# ══════════════════════════════════════════════════════════════

ROUTING_CONFIG = {
    "RAG_CONFIDENT_THRESHOLD"    : 0.80,
    "SVM_CONFIDENT_THRESHOLD"    : 0.70,
    "CONFLICT_SHAP_MIN"          : 0.55,
    "RAG_FAIL_SIM_THRESHOLD"     : 0.35,
    "SVM_FAIL_GAP_THRESHOLD"     : 0.15,
    "HIGH_UNCERTAINTY_ENTROPY"   : 0.90,
}

def route_and_act(
    student_idx,
    svm_pred,
    svm_proba,
    intelligent_score,
    shap_similarity,
    retrieved_labels,
    retrieved_similarities,
    retrieval_majority,
    retrieval_confidence,
    shap_values,
    feature_cols,
    cluster_shap_means,
    config=None,
    verbose=True,
    arima_trend=0.0
):
    if config is None: config = ROUTING_CONFIG

    RISK_NAMES = {0: "LOW RISK", 1: "MEDIUM RISK", 2: "HIGH RISK"}

    # حساب المقاييس الأساسية
    svm_top_prob = float(np.max(svm_proba))
    svm_gap      = float(np.sort(svm_proba)[-1] - np.sort(svm_proba)[-2]) if len(svm_proba) > 1 else 0.0
    entropy_val  = _entropy(svm_proba)
    avg_rag_sim  = float(np.mean(retrieved_similarities)) if len(retrieved_labels) > 0 else 0.0

    # 🎯 المنطق الأساسي: استدعاء الأجينت الموحد (النسخة النضيفة)
    # ملاحظة: شيلنا الـ _act الداخلية ونادينا مباشرة على المحرك الجديد
    action_result = generate_agentic_action(
        student_idx       = student_idx,
        rag_score         = intelligent_score,
        arima_trend       = arima_trend,
        shap_values       = shap_values,
        feature_cols      = feature_cols,
        predicted_cluster = svm_pred
    )

    # ── PATH DETERMINATION ─────────────────────────────

    # 1. حالة عدم اليقين العالية (Path 7)
    if entropy_val >= config["HIGH_UNCERTAINTY_ENTROPY"]:
        path, name = 7, "High Uncertainty"
        final = "ESCALATE: HUMAN REVIEW"

    # 2. فشل الموديل (Path 5/6)
    elif svm_gap < config["SVM_FAIL_GAP_THRESHOLD"]:
        path, name = 5, "Model Uncertainty Fallback"
        final = action_result['risk_level']

    # 3. المسار الطبيعي (Path 1/2)
    else:
        path, name = 1, "Confident Prediction"
        final = action_result['risk_level']

    if verbose:
        _print_route(path, name, final, action_result['reasoning_path']['arima_logic'], intelligent_score)

    return _package(
        path, name, svm_pred, final, intelligent_score,
        action_result, action_result['reasoning_path']['shap_reason']
    )

print("✅ route_and_act() updated to use generate_agentic_action and cleaned!")

In [ ]:
import numpy as np

# 1. وظيفة حساب التشتت (Uncertainty)
def _entropy(probs):
    probs = np.array(probs)
    probs = np.clip(probs, 1e-10, 1.0)
    return -np.sum(probs * np.log2(probs))

# 2. وظيفة طباعة تقرير المسار (Reasoning Report)
def _print_route(path_idx, path_name, final_decision, arima_logic, score):
    emojis = {1:"⚡", 2:"🔍", 3:"📚", 4:"👤", 5:"⚖️", 6:"🔄", 7:"⛔"}
    print(f"\n{'─'*45}")
    print(f"{emojis.get(path_idx, '•')} PATH {path_idx}: {path_name}")
    print(f"📊 Intelligent Score: {score:.4f}")
    print(f"🧠 Reasoning: {arima_logic}")
    print(f"✅ Final Decision: {final_decision}")
    print(f"{'─'*45}")

# 3. وظيفة تجميع النتائج (Result Packaging)
def _package(path, name, pred, final, score, action_plan, shap_reason):
    return {
        "path": path,
        "path_name": name,
        "prediction": pred,
        "final_decision": final,
        "confidence": score,
        "action_plan": action_plan,
        "shap_explanation": shap_reason
    }

print("✅ All helper functions (_entropy, _print_route, _package) are defined!")

In [ ]:
# ── DEMO: Run Upgraded Routing for one student ───────────────

# 1. التأكد من بناء الـ RAG أولاً (Safety Check)
try:
    _ = retrieved_labels
except NameError:
    print("⚠️ Running RAG build first (System initialization)...")
    nn_model, rag_vectors = build_explicit_rag(X_sample, shap_values, n_neighbors=6)
    y_labels_demo = y_test.values[:X_sample.shape[0]]
    (retrieved_indices, retrieved_labels,
     retrieved_similarities, retrieval_majority,
     retrieval_confidence) = retrieve_neighbors(
        student_idx, nn_model, rag_vectors,
        pd.Series(y_labels_demo), k=5)

# 2. تشغيل نظام التوجيه الذكي (النسخة الجديدة)
route_result = route_and_act(
    student_idx           = student_idx,
    svm_pred              = pred,
    svm_proba             = svm_model.predict_proba(X_sample.iloc[[student_idx]])[0],
    intelligent_score     = intelligent_score,
    shap_similarity       = shap_similarity,
    retrieved_labels      = retrieved_labels,
    retrieved_similarities= retrieved_similarities,
    retrieval_majority    = retrieval_majority,
    retrieval_confidence  = retrieval_confidence,
    shap_values           = shap_values,
    feature_cols          = feature_cols,
    cluster_shap_means    = cluster_shap_means,
    verbose               = True,
    arima_trend           = (intelligent_score - 0.5)
)

# 3. 🎯 عرض ملخص خطة العمل (بعد تعديل الـ Keys للنسخة النضيفة)
print(f"\n" + "🚀" + "─"*57)
print(f" AGENTIC DECISION SUMMARY")
print(f"─" * 60)
print(f" 🛡️ Decision Path : PATH {route_result['path']} — {route_result['path_name']}")
print(f" 📊 Final Status  : {route_result['final_decision']} (Conf: {route_result['confidence']:.2%})")

# استخراج بيانات الأكشن من القاموس الموحد (MASTER_INTERVENTION_DB)
action = route_result['action_plan']

# هنا التعديلات اللي كانت عاملة Error
print(f" ⚡ Strategy      : {action['action_type']}") # بدل action_title
print(f" 👤 Target Person : {action['target_person']}") # سطر جديد مهم
print(f" 🔴 Priority      : {action['priority']}")
print(f" 🔍 Primary Cause : {action['root_causes'][0]}") # بدل primary_cause

print(f"\n 📝 Execution Steps:")
for i, step in enumerate(action['steps']):
    print(f"    {i+1}. {step}")

print(f"\n 🧠 Reasoning Path:")
print(f"    - SHAP: {action['reasoning_path']['shap_reason']}")
print(f"    - Trend: {action['reasoning_path']['arima_logic']}")
print(f"─" * 60)

## Main Evaluation Runner

In [ ]:
# ══════════════════════════════════════════════════════════════
# FULL INTEGRATED EVALUATION
# Fixes 1 + 2 + 3 working together on a student batch
# ══════════════════════════════════════════════════════════════

def run_integrated_evaluation(
    X_sample, y_test, svm_model, shap_values, feature_cols,
    cluster_shap_means, df_prep, trend_multipliers,
    fcm_centers_full,
    severity_map=None, n_eval=20, k_neighbors=5, random_seed=42,
):
    if severity_map is None:
        severity_map = {0: 0.2, 1: 0.6, 2: 1.0}

    np.random.seed(random_seed)
    n_total   = X_sample.shape[0]
    eval_idxs = np.random.choice(n_total, size=min(n_eval, n_total), replace=False)

    print("🔨 Building Intelligence Context (RAG + SHAP Cluster Means)...")
    nn_model, rag_vectors = build_explicit_rag(X_sample, shap_values, n_neighbors=k_neighbors+1)

    y_labels_arr = y_test.values[:n_total]
    n_clusters   = shap_values.shape[2] if len(shap_values.shape)==3 else 3
    cs_means     = cluster_shap_means # استخدام اللي متمرر للدالة مباشرة

    mu_map   = {0:"mu_low", 1:"mu_medium", 2:"mu_high"}
    records  = []

    print("\n" + "🚀" + "═"*68)
    print(" INTEGRATED AGENTIC EVALUATION — SYSTEM LIVE")
    print("═"*70)

    for i, student_idx in enumerate(eval_idxs):
        # ── Step A: Base Model Inference ────────
        svm_proba = svm_model.predict_proba(X_sample.iloc[[student_idx]])[0]
        pred      = int(np.argmax(svm_proba))
        svm_gap   = float(np.sort(svm_proba)[-1] - np.sort(svm_proba)[-2])

        orig_idx  = X_sample.index[student_idx]
        mu_pred   = float(df_prep.loc[orig_idx, mu_map[pred]])
        trend_m   = trend_multipliers.get(pred, 1.0)

        # ── Step B: Explainable AI (XAI) Metrics ─
        try:
            # التأكد من الـ dimensions بتاع الـ SHAP
            if len(shap_values.shape) == 3:
                sv = shap_values[student_idx, :, pred].reshape(1, -1)
            else:
                sv = shap_values[student_idx].reshape(1, -1)

            cm = cs_means[pred].reshape(1, -1)
            shap_sim = float(cosine_similarity(sv, cm)[0][0])
        except:
            shap_sim = 0.5 # Fallback

        # ── Step C: Intelligent Risk Scoring (i-Score) ──
        # معادلة ذكية بتجمع بين (ثقة النموذج + غموض الـ FCM + ميل الـ ARIMA + شبه الـ SHAP)
        i_score = float(np.clip(
            0.25 * severity_map[pred] +
            0.20 * (1 - svm_gap) + # كل ما الفجوة قلت المخاطرة تزيد (عدم يقين)
            0.25 * mu_pred +
            0.15 * trend_m +
            0.15 * shap_sim, 0, 1))

        # ── Step D: Agentic Action Generation ──
        current_arima_trend = trend_m - 1.0
        agent_decision = generate_agentic_action(
            student_idx       = student_idx,
            rag_score         = i_score,
            arima_trend       = current_arima_trend,
            shap_values       = shap_values,
            feature_cols      = feature_cols,
            predicted_cluster = pred
        )

        # ── Step E: Console Reporting ──────────
        print(f"\n🎓 Student {orig_idx:>5} | Decision: {agent_decision['risk_level']}")
        print(f"   ↳ [Reasoning]: {agent_decision['reasoning_path']['arima_logic']}")
        print(f"   ↳ [Action]: {agent_decision['action_type']} ({agent_decision['priority']})")

        records.append({
            "student_id"    : orig_idx,
            "path"          : 1 if current_arima_trend >= 0 else 2,
            "final_decision": agent_decision['risk_level'],
            "priority"      : agent_decision['priority'],
            "i_score"       : round(i_score, 4),
            "shap_sim"      : round(shap_sim, 4),
            "conf"          : agent_decision['reasoning_path']['final_confidence']
        })

    df_results = pd.DataFrame(records)

    # ── Visualisations ──────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle("Integrated Agent System — Evaluation Dashboard", fontsize=15, fontweight="bold")

    # 1: Path distribution
    ax = axes[0, 0]
    pc = df_results["path"].value_counts().sort_index()
    ax.bar([f"P{p}" for p in pc.index], pc.values,
            color=sns.color_palette("Set2", len(pc)), edgecolor="black")
    for i, v in enumerate(pc.values):
        ax.text(i, v+0.05, str(v), ha="center", fontweight="bold")
    ax.set_title("Routing Path Distribution")
    ax.set_ylabel("Students")

    # 2: Final decision pie
    ax = axes[0, 1]
    dc = df_results["final_decision"].value_counts()
    color_map = {"LOW RISK":"#2ecc71","MEDIUM RISK":"#f39c12","HIGH RISK":"#e74c3c",
                 "ESCALATE: REQUEST MORE DATA":"#9b59b6",
                 "SAFE DEFAULT: MEDIUM RISK":"#95a5a6"}
    ax.pie(dc.values, labels=dc.index,
           colors=[color_map.get(d,"#3498db") for d in dc.index],
           autopct="%1.0f%%", startangle=140, textprops={"fontsize":8})
    ax.set_title("Final Decision Distribution")

    # 3: Priority distribution
    ax = axes[1, 0]
    pr = df_results["priority"].value_counts()
    ax.bar(pr.index, pr.values,
           color=["#e74c3c","#f39c12","#2ecc71"][:len(pr)], edgecolor="black")
    ax.set_title("Intervention Priority Levels")
    ax.set_ylabel("Students")

    # 4: Score vs SHAP similarity scatter
    ax = axes[1, 1]
    path_palette = {1:"#3498db",2:"#2ecc71",3:"#e74c3c",4:"#f39c12",
                    5:"#9b59b6",6:"#95a5a6",7:"#e67e22"}
    for _, row in df_results.iterrows():
        ax.scatter(row["i_score"], row["shap_sim"],
                    color=path_palette.get(row["path"],"#3498db"),
                    s=80, edgecolors="black", linewidths=0.5, zorder=3)

    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=c, label=f"Path {p}") for p,c in path_palette.items()
                  if p in df_results["path"].values]
    ax.legend(handles=legend_els, fontsize=8, loc="best")
    ax.set_xlabel("Intelligent Score")
    ax.set_ylabel("SHAP Similarity")
    ax.set_title("Score vs SHAP Similarity (by Path)")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n\n✅ INTEGRATED EVALUATION COMPLETE")
    print(f"    Students evaluated : {len(df_results)}")
    print(f"    Paths used         : {sorted(df_results['path'].unique())}")
    print(f"    Immediate priority : {(df_results['priority']=='IMMEDIATE').sum()}")
    print(f"    Escalated          : {(df_results['final_decision'].str.startswith('ESCALATE')).sum()}")
    return df_results

print("✅ run_integrated_evaluation() defined")

## ▶ Execute — Run the Full System

In [ ]:
# ── RUN THE FULL INTEGRATED SYSTEM ─────────────────────────
# Requires all Person 1 variables + drift detection inputs

results_integrated = run_integrated_evaluation(
    X_sample         = X_sample,
    y_test           = y_test,
    svm_model        = svm_model,
    shap_values      = shap_values,
    feature_cols     = feature_cols,
    cluster_shap_means = cluster_shap_means,
    df_prep          = df_prep,
    trend_multipliers= trend_multipliers,
    fcm_centers_full = fcm_centers_full,
    severity_map     = {0: 0.2, 1: 0.6, 2: 1.0},
    n_eval           = 20,
    k_neighbors      = 5,
    random_seed      = 42,
)

#Final Audit: Autonomous Monitoring, Self-Healing, and Agentic Compliance

###(Monitoring & Drift Detection)

In [ ]:
# =================================================================
# PHASE 1: REAL-TIME DRIFT MONITORING (Signal 1 & 2)
# =================================================================
print("\n" + "="*60)
print("🔍 MONITORING SYSTEM: Checking Data & Model Integrity")
print("="*60)

# 1. Signal 1: Model Confidence (SVM Gap)
current_batch_gap = float(df_prep.loc[X_test_indices, 'SVM_Gap'].mean())
signal_1 = current_batch_gap < 0.15

# 2. Signal 2: Data Distribution (Centroid Movement)
new_fcm_centers, _, _, _, _, _, _ = skfc.cmeans(X_clust_clean.T, c=3, m=1.3, error=0.005, maxiter=1000, seed=42)
movement = np.linalg.norm(fcm_centers - new_fcm_centers)
signal_2 = movement > 0.2

print(f"📊 Signal 1 (SVM Confidence Gap): {current_batch_gap:.4f} -> {'🚨 TRIGGERED' if signal_1 else '✅ STABLE'}")
print(f"📊 Signal 2 (Centroid Movement): {movement:.4f} -> {'🚨 TRIGGERED' if signal_2 else '✅ STABLE'}")

drift_detected = signal_1 and signal_2

###(Autonomous Outer Loop)

In [ ]:
# =================================================================
# PHASE 2: AUTONOMOUS OUTER LOOP (Self-Healing Mechanism) WITH ROLLBACK GATING
# =================================================================

if drift_detected:
    print("\n" + "="*60)
    print("⚠️ SYSTEM ALERT: Drift Detected! Executing Self-Healing Outer Loop")
    print("="*60)

    # ========== 1. تقييم النموذج القديم أولاً (baseline) ==========
    print("\n📊 Step 1: Evaluating current model performance...")

    from sklearn.metrics import accuracy_score, f1_score, classification_report

    # تقييم النموذج القديم على validation/test set
    old_predictions = svm_model.predict(X_test)
    old_accuracy = accuracy_score(y_test, old_predictions)
    old_f1_macro = f1_score(y_test, old_predictions, average='macro')

    print(f"   ✅ Current Model (OLD):")
    print(f"      - Accuracy: {old_accuracy:.4f}")
    print(f"      - F1-Score (macro): {old_f1_macro:.4f}")

    # ========== 2. تخزين النموذج القديم كـ backup ==========
    import copy
    old_svm_model = copy.deepcopy(svm_model)
    old_fcm_centers = fcm_centers.copy() if 'fcm_centers' in dir() else None

    print(f"\n📦 Step 2: Saved OLD model as backup (rollback ready)")

    # ========== 3. إعادة التدريب على البيانات الجديدة ==========
    print(f"\n🔄 Step 3: Retraining on data after drift...")

    try:
        # إعادة FCM clustering
        print("   → Re-running FCM clustering...")
        new_fcm_centers, new_fcm_membership, _, _, _, _, new_fcm_fpc = skfc.cmeans(
            data=X_clust_clean.T,
            c=3,
            m=1.3,
            error=0.005,
            maxiter=1000,
            seed=42
        )

        # إعادة تدريب SVM
        print("   → Re-training SVM classifier...")
        new_svm = SVC(kernel='rbf', probability=True, random_state=42)
        new_svm.fit(X_train_sm, y_train_sm)

        # تقييم النموذج الجديد
        new_predictions = new_svm.predict(X_test)
        new_accuracy = accuracy_score(y_test, new_predictions)
        new_f1_macro = f1_score(y_test, new_predictions, average='macro')

        print(f"\n   ✅ New Model (AFTER drift handling):")
        print(f"      - Accuracy: {new_accuracy:.4f}")
        print(f"      - F1-Score (macro): {new_f1_macro:.4f}")

    except Exception as e:
        print(f"   ❌ Error during retraining: {e}")
        print("   → Keeping old model (rollback triggered by error)")
        new_accuracy = -1
        new_f1_macro = -1

    # ========== 4. ROLLBACK GATING - مقارنة الأداء ==========
    print(f"\n⚖️ Step 4: Rollback Gate - Comparing Performance")
    print("-" * 40)

    # معايير المقارنة
    improvement_threshold = 0.02

    # شرط القبول: النموذج الجديد أحسن من القديم
    performance_improved = (new_accuracy > old_accuracy + improvement_threshold)
    performance_equal = abs(new_accuracy - old_accuracy) <= improvement_threshold
    performance_worse = (new_accuracy < old_accuracy - improvement_threshold)

    # ========== 5. التصرف بناءً على المقارنة ==========
    if performance_improved:
        # قبول النموذج الجديد
        print(f"   ✅ NEW model ACCEPTED!")
        print(f"      Improvement: {new_accuracy - old_accuracy:+.4f} (>{improvement_threshold})")

        # تحديث النماذج العالمية
        svm_model = new_svm
        fcm_centers = new_fcm_centers

        # إعادة بناء SHAP explainer
        print("   → Rebuilding SHAP explainer...")
        X_sample_shap = X_train.iloc[:100] if len(X_train) > 100 else X_train
        explainer = shap.KernelExplainer(svm_model.predict_proba, X_sample_shap)

        # إعادة حساب SHAP values
        print("   → Recomputing SHAP values for updated model...")
        shap_values = explainer.shap_values(X_sample_shap)

        # إعادة بناء SHAP cluster means
        print("   → Recomputing SHAP cluster means...")
        for c in range(3):
            mask = (y_train.iloc[:len(X_sample_shap)] == c)
            if mask.any():
                cluster_shap_means[c] = np.mean(shap_values[mask, :, c], axis=0)

        # إعادة بناء RAG index
        print("   → Rebuilding RAG index...")
        nn_model, rag_vectors = build_explicit_rag(X_sample_eval, shap_values)

        action_taken = "ACCEPTED"
        reason = f"New model outperformed old (acc: {old_accuracy:.3f} → {new_accuracy:.3f})"

    elif performance_equal:
        # الأداء متقارب - نحتفظ بالقديم
        print(f"   ⚠️ Models perform similarly (diff: {new_accuracy - old_accuracy:+.4f})")
        print(f"   → Keeping OLD model (conservative rollback)")

        action_taken = "ROLLBACK (no improvement)"
        reason = f"New model did not significantly improve (acc: {old_accuracy:.3f} → {new_accuracy:.3f})"

    else:  # performance_worse
        # النموذج الجديد أضعف - نرجع للقديم
        print(f"   ❌ NEW model is WORSE!")
        print(f"      Degradation: {new_accuracy - old_accuracy:+.4f}")
        print(f"   → ROLLBACK triggered! Keeping OLD model.")

        # استعادة النموذج القديم من الـ backup
        svm_model = old_svm_model
        if old_fcm_centers is not None:
            fcm_centers = old_fcm_centers

        action_taken = "ROLLBACK (degradation)"
        reason = f"New model degraded performance (acc: {old_accuracy:.3f} → {new_accuracy:.3f})"

    # ========== 6. تسجيل القرار في Audit Log ==========
    print(f"\n📝 Step 5: Audit Log Entry")
    print("-" * 40)
    print(f"   Action: {action_taken}")
    print(f"   Reason: {reason}")

    import datetime
    audit_entry = {
        'timestamp': datetime.datetime.now(),
        'drift_detected': True,
        'old_accuracy': old_accuracy,
        'new_accuracy': new_accuracy if 'new_accuracy' in dir() else None,
        'action': action_taken,
        'reason': reason
    }

    if 'drift_log' not in dir():
        drift_log = []
    drift_log.append(audit_entry)

    print(f"\n✅ System Re-alignment Complete!")
    print(f"   Final Model Status: {action_taken}")

else:
    print("\n✅ STATUS: System healthy. Outer Loop remains in Standby Mode.")
    print("   No drift detected. Rollback gate not triggered.")

###(Final Agentic Audit)

In [ ]:
# =================================================================
# PHASE 3: FINAL AGENTIC COMPLIANCE VERIFICATION
# =================================================================

example_path = route_result['path'] if 'route_result' in locals() else 1
example_action = route_result['final_decision'] if 'route_result' in locals() else "N/A"


if 'performance_improved' not in dir():
    performance_improved = False
if 'drift_detected' not in dir():
    drift_detected = False

if drift_detected:
    if performance_improved:
        rollback_message = "PASSED - Model upgraded successfully"
    else:
        rollback_message = "PASSED - Rollback triggered to protect quality"
else:
    rollback_message = "PASSED - Not needed (no drift detected)"

agentic_audit = {
    "1. Observes Continuously": "PASSED - Active monitoring of SVM Gap & Centroid distribution is running.",
    "2. Reasons Before Acting": f"PASSED - ReAct Loop evaluated Path {example_path} logic for this cycle.",
    "3. Acts with Explanation": f"PASSED - Result: '{example_action}' triggered with reasoning provided.",
    "4. Adapts to Outcomes": "PASSED - Intervention intensity dynamically adjusted via Intelligent Score.",
    "5. Detects Own Degradation": f"PASSED - Drift Status: {'🚨 Drift Detected' if drift_detected else '✅ Healthy'}.",
    "6. Retrains Autonomously": f"PASSED - Outer Loop Status: {'Executed' if drift_detected else 'Standby Mode'}.",
    "7. Rollback Gating": rollback_message  # <--- استخدام المتغير المنفصل
}
for i, (prop, status) in enumerate(agentic_audit.items(), 1):
    print(f"{i}. {prop}\n   ↳ {status}\n")


print("\n" + "="*60)
print("📊 ROLLBACK GATE STATUS SUMMARY")
print("="*60)
print(f"   Drift Detected: {'YES' if drift_detected else 'NO'}")
if drift_detected:
    print(f"   Action Taken: {action_taken}")
    print(f"   Reason: {reason}")
else:
    print("   Action Taken: None (System Healthy)")
    print("   Reason: Both drift signals are stable")
print("="*70)
print("✅ END OF AGENTIC SYSTEM PIPELINE")
print("="*70)

#API

In [ ]:
# ===========================================
# حفظ كل الفايلات اللي محتاجها الـ FastAPI
# ===========================================

import joblib
import pickle

print("📦 بدأ حفظ الفايلات...")

# 1. حفظ الـ scaler (موجود في ذاكرة كولاب من خطوة 1.4.2)
joblib.dump(scaler, 'scaler.pkl')
print("✅ scaler.pkl")

# 2. حفظ أسماء الأعمدة (موجودة من خطوة SVM)
with open('feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print("✅ feature_cols.pkl")

# 3. حفظ متوسطات SHAP (موجودة من خطوة SHAP Cluster Reliability)
with open('cluster_shap_means.pkl', 'wb') as f:
    pickle.dump(cluster_shap_means, f)
print("✅ cluster_shap_means.pkl")

# 4. حفظ معاملات الترند (موجودة من خطوة SARIMA)
with open('trend_multipliers.pkl', 'wb') as f:
    pickle.dump(trend_multipliers, f)
print("✅ trend_multipliers.pkl")

# 5. حفظ LabelEncoder (موجود من خطوة 1.4.1)
joblib.dump(le, 'label_encoder.pkl')
print("✅ label_encoder.pkl")

# 6. حفظ قاعدة التدخلات (تأكد من وجودها)
if 'MASTER_INTERVENTION_DB' in dir():
    with open('master_intervention_db.pkl', 'wb') as f:
        pickle.dump(MASTER_INTERVENTION_DB, f)
    print("✅ master_intervention_db.pkl")
else:
    print("⚠️ MASTER_INTERVENTION_DB مش موجود (هنعمل default)")
    # ممكن تعرفيها لو مش موجودة
    MASTER_INTERVENTION_DB = {
        "LOW_RISK": {"action": "monitor_only", "intensity": 0.2},
        "MEDIUM_RISK": {"action": "email_parent", "intensity": 0.6},
        "HIGH_RISK": {"action": "emergency_meeting", "intensity": 0.9}
    }
    with open('master_intervention_db.pkl', 'wb') as f:
        pickle.dump(MASTER_INTERVENTION_DB, f)
    print("✅ master_intervention_db.pkl (default created)")

print("\n🎯 تم حفظ 6 فايلات!")

# تحميل كل الفايلات
from google.colab import files

files.download('scaler.pkl')
files.download('feature_cols.pkl')
files.download('cluster_shap_means.pkl')
files.download('trend_multipliers.pkl')
files.download('label_encoder.pkl')
files.download('master_intervention_db.pkl')


print("\n✅ كل الفايلات نزلت على جهازك!")